# Destabilizing Mutation → Protein Abundance Analysis\n\nTests whether destabilizing missense mutations reduce protein abundance using shared peptides from psm.tsv.\n\n**Logic:**  \nFor each destabilizing mutation (high AlphaMissense OR SPURS ddG) in protein X within a plex:\n1. Identify which TMT channels carry the mutation (GDC UUID → missense table)\n2. Find **shared peptides**: reference PSMs mapping to protein X (not -mut, not -comp)\n3. Compare their TMT intensities: mutant-carrier channels vs non-carrier channels in same plex\n4. `delta = log2(mut_channel / mean(WT_channels))` — negative = less abundant\n\nA **neutral control** (benign by AM + SPURS) runs identically as negative control."

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP      = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META     = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
MISSENSE     = "/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv"
RESULTS_BASE = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/results"
PLEX_LIST    = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
OUT_DIR      = "/scratch/leduc.an/AAS_Evo/ANALYSIS/destabilization"

# Destabilizing thresholds
AM_THRESHOLD    = 0.564
SPURS_THRESHOLD = 0.5
VAF_THRESHOLD   = 0.3
GNOMAD_MAX      = 0.01

# Neutral control thresholds
AM_BENIGN_MAX    = 0.34
SPURS_BENIGN_MAX = 0.0

N_PLEXES = None   # None = all; set int to test on subset e.g. 5

CHANNEL_ORDER = ["126","127N","127C","128N","128C","129N","129C","130N","130C","131N","131C"]
TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
    "tmt_126c":"126C","tmt_134n":"134N",
}

os.makedirs(OUT_DIR, exist_ok=True)
print("Config loaded")

In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
if N_PLEXES:
    plex_ids = plex_ids[:N_PLEXES]

print(f"Plexes: {len(plex_ids)} | TMT map: {len(tmt):,} rows | GDC meta: {len(gdc):,} rows")

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print("Loading missense table...")
missense = pd.read_csv(MISSENSE, sep="\t", low_memory=False)
missense["am_pathogenicity"] = pd.to_numeric(missense["am_pathogenicity"], errors="coerce")
missense["spurs_ddg"]        = pd.to_numeric(missense["spurs_ddg"], errors="coerce")
missense["VAF"]              = pd.to_numeric(missense["VAF"], errors="coerce")
missense["gnomADe_AF"]       = pd.to_numeric(missense["gnomADe_AF"], errors="coerce").fillna(0)
print(f"Total rows: {len(missense):,}")

base_ok = (missense["VAF"] >= VAF_THRESHOLD) & (missense["gnomADe_AF"] < GNOMAD_MAX)

destab = missense[((missense["am_pathogenicity"] >= AM_THRESHOLD) |
                   (missense["spurs_ddg"] >= SPURS_THRESHOLD)) & base_ok].copy()

neutral = missense[(missense["am_pathogenicity"] <= AM_BENIGN_MAX) &
                   (missense["spurs_ddg"] <= SPURS_BENIGN_MAX) &
                   missense["spurs_ddg"].notna() & base_ok].copy()

# sample_id (GDC UUID) -> set of gene symbols with mutations
destab_sample_genes  = destab.groupby("sample_id")["SYMBOL"].apply(set).to_dict()
neutral_sample_genes = neutral.groupby("sample_id")["SYMBOL"].apply(set).to_dict()

print(f"Destabilizing: {len(destab):,} rows, {destab['SYMBOL'].nunique()} genes, {len(destab_sample_genes)} samples")
print(f"Neutral:       {len(neutral):,} rows, {neutral['SYMBOL'].nunique()} genes, {len(neutral_sample_genes)} samples")

In [ ]:
# ── HELPERS ───────────────────────────────────────────────────────────────────
def find_ri_cols(df):
    """Returns dict channel_label -> column_name for TMT intensity columns."""
    found = {}
    all_channels = CHANNEL_ORDER + ["126C","132N","132C","133N","133C","134N"]
    for ch in all_channels:
        if ch in df.columns:
            found[ch] = ch; continue
        candidates = [c for c in df.columns if c.startswith("Intensity") and c.endswith(f"_{ch}")]
        if candidates:
            found[ch] = candidates[0]
    return found

def get_channel_uuid_map(plex_id):
    """Returns dict: TMT channel label -> gdc_file_id."""
    pt = tmt[tmt["run_metadata_id"] == plex_id][["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates()
    pt = pt[~pt["case_submitter_id"].str.lower().isin(["ref","reference","pooled","pool","nan"])]
    pt["channel"] = pt["tmt_channel"].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=["channel"])
    merged = pt.merge(gdc[["gdc_file_id","case_submitter_id","sample_type"]],
                      on=["case_submitter_id","sample_type"], how="left")
    return merged.dropna(subset=["gdc_file_id"]).drop_duplicates("channel").set_index("channel")["gdc_file_id"].to_dict()

def gene_from_entry(entry_name):
    """Gene symbol from PSM Entry Name (GENE_HUMAN format). Returns None for -mut/-comp."""
    if pd.isna(entry_name):
        return None
    s = str(entry_name)
    if s.endswith(("-mut", "-comp")):
        return None
    m = re.match(r'^([A-Z0-9]+)_HUMAN', s)
    return m.group(1) if m else None

print("Helpers defined")

In [ ]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
# For each plex, load psm.tsv.
# For each reference PSM (not -mut/-comp), get gene from Entry Name.
# For channels carrying a destabilizing mutation in that gene:
#   delta = log2(mut_channel_intensity / mean(WT_channel_intensities))

def run_loop(sample_gene_map, label="destabilizing"):
    records = []
    n_done = 0

    for plex_id in plex_ids:
        results_dir = os.path.join(RESULTS_BASE, plex_id)
        psm_files = sorted(glob.glob(os.path.join(results_dir, "*_1/psm.tsv")))
        if not psm_files:
            psm_files = sorted(glob.glob(os.path.join(results_dir, "psm.tsv")))
        if not psm_files:
            continue

        try:
            psm = pd.read_csv(psm_files[0], sep="\t", low_memory=False)
        except Exception:
            continue

        ri_col_map = find_ri_cols(psm)
        if not ri_col_map:
            continue

        ch_uuid = get_channel_uuid_map(plex_id)
        ri_channels = list(ri_col_map.keys())
        plex_chs = [ch for ch in ri_channels if ch in ch_uuid]

        # Reference PSMs only
        ref_mask = ~psm["Entry Name"].astype(str).str.endswith(("-mut", "-comp"))
        ref_psm = psm[ref_mask].copy()
        ref_psm["_gene"] = ref_psm["Entry Name"].apply(gene_from_entry)
        ref_psm = ref_psm.dropna(subset=["_gene"])

        for _, row in ref_psm.iterrows():
            gene = row["_gene"]

            # Channels in this plex that carry a mutation in this gene
            mut_chs = [ch for ch in plex_chs
                       if gene in sample_gene_map.get(ch_uuid.get(ch, ""), set())]
            if not mut_chs:
                continue

            # WT channels = in plex but no mutation in this gene
            wt_chs = [ch for ch in plex_chs
                      if ch not in mut_chs
                      and gene not in sample_gene_map.get(ch_uuid.get(ch, ""), set())]
            if len(wt_chs) < 2:
                continue

            wt_vals = [row[ri_col_map[ch]] for ch in wt_chs
                       if pd.notna(row[ri_col_map[ch]]) and row[ri_col_map[ch]] > 0]
            if len(wt_vals) < 2:
                continue
            wt_mean = np.mean(wt_vals)

            for mut_ch in mut_chs:
                v = row[ri_col_map[mut_ch]]
                if pd.isna(v) or v <= 0:
                    continue
                records.append({
                    "plex_id": plex_id,
                    "gene":    gene,
                    "channel": mut_ch,
                    "peptide": row.get("Peptide", ""),
                    "delta":   np.log2(v / wt_mean),
                    "wt_n":    len(wt_vals),
                })

        n_done += 1
        if n_done % 10 == 0:
            print(f"  [{label}] {n_done} plexes, {len(records):,} records")

    print(f"[{label}] Done: {n_done} plexes, {len(records):,} records")
    return pd.DataFrame(records)

print("Running destabilizing loop...")
df = run_loop(destab_sample_genes, "destabilizing")
print("\nRunning neutral loop...")
df_neutral = run_loop(neutral_sample_genes, "neutral")

In [ ]:
# ── GLOBAL STATISTICS ─────────────────────────────────────────────────────────
t,   p   = stats.ttest_1samp(df["delta"].dropna(),         popmean=0)
t_n, p_n = stats.ttest_1samp(df_neutral["delta"].dropna(), popmean=0)
u,   p_mw = stats.mannwhitneyu(df["delta"].dropna(), df_neutral["delta"].dropna(), alternative="less")

print(f"Destabilizing: n={len(df):,}  mean={df['delta'].mean():.4f}  t={t:.2f}  p={p:.2e}")
print(f"Neutral:       n={len(df_neutral):,}  mean={df_neutral['delta'].mean():.4f}  t={t_n:.2f}  p={p_n:.2e}")
print(f"Mann-Whitney U (destabilizing < neutral): p={p_mw:.2e}")

In [ ]:
# ── PLOT: side-by-side delta distributions ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

q01 = min(df["delta"].quantile(0.01), df_neutral["delta"].quantile(0.01))
q99 = max(df["delta"].quantile(0.99), df_neutral["delta"].quantile(0.99))

for ax, data, label, color, t_val, p_val in [
    (axes[0], df,         f"Destabilizing\n(AM≥{AM_THRESHOLD} OR SPURS≥{SPURS_THRESHOLD})", "#d62728", t,   p),
    (axes[1], df_neutral, f"Neutral\n(AM≤{AM_BENIGN_MAX} AND SPURS≤{SPURS_BENIGN_MAX})",    "#2ca02c", t_n, p_n),
]:
    ax.hist(data["delta"], bins=80, color=color, alpha=0.75, edgecolor="none", range=(q01, q99))
    mean_d = data["delta"].mean()
    ax.axvline(0, color="k", lw=1, ls="--")
    ax.axvline(mean_d, color="k", lw=1.5, ls="-",
               label=f"mean={mean_d:.3f}\nt={t_val:.2f}, p={p_val:.1e}")
    ax.set_xlabel("log2(shared peptide intensity: mut channel / plex WT mean)")
    ax.set_ylabel("PSM count")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle(f"Shared peptide abundance: destabilizing vs neutral mutations\nMann-Whitney p={p_mw:.2e}", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "destab_vs_neutral_dist.pdf"), dpi=150)
plt.show()

In [ ]:
# ── PER-GENE STATS (destabilizing) ────────────────────────────────────────────
from statsmodels.stats.multitest import multipletests

gene_stats = []
for gene, grp in df.groupby("gene"):
    deltas = grp["delta"].dropna().values
    if len(deltas) < 3:
        continue
    t_g, p_g = stats.ttest_1samp(deltas, popmean=0)
    gene_stats.append({"gene": gene, "n_psms": len(deltas), "n_plexes": grp["plex_id"].nunique(),
                       "mean_delta": np.mean(deltas), "t_stat": t_g, "p_value": p_g})

gene_stats = pd.DataFrame(gene_stats).sort_values("p_value")
if len(gene_stats):
    _, gene_stats["fdr"], _, _ = multipletests(gene_stats["p_value"], method="fdr_bh")

sig = (gene_stats["fdr"] < 0.05) & (gene_stats["mean_delta"] < 0)
print(f"Genes tested: {len(gene_stats)} | Significant (FDR<0.05, delta<0): {sig.sum()}")
gene_stats.head(20)

In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(OUT_DIR, "destab_per_psm.tsv"), sep="\t", index=False)
df_neutral.to_csv(os.path.join(OUT_DIR, "neutral_per_psm.tsv"), sep="\t", index=False)
gene_stats.to_csv(os.path.join(OUT_DIR, "gene_destab_stats.tsv"), sep="\t", index=False)
print(f"Saved to {OUT_DIR}")

In [ ]:
# ── NEUTRAL CONTROL LOOP ──────────────────────────────────────────────────────
records_neutral = []

for plex_id in plex_ids:
    results_dir = os.path.join(RESULTS_BASE, plex_id)
    abund_files = glob.glob(os.path.join(results_dir, "**/abundance_protein_MD.tsv"), recursive=True)
    if not abund_files:
        abund_files = glob.glob(os.path.join(results_dir, "tmt-report/abundance_protein_MD.tsv"))
    if not abund_files:
        continue

    abund = pd.read_csv(abund_files[0], sep="\t", index_col=0)
    annot_cols = {"NumberPSM","Protein","Gene","ProteinID","Entry","MaxPepProb","ReferenceIntensity"}
    ch_cols = [c for c in abund.columns if c not in annot_cols and abund[c].dtype in [float, "float64"]]
    ch_uuid = get_channel_uuid_map(plex_id)
    gene_col = next((c for c in abund.columns if c.lower() in ["gene","genename","symbol"]), None)

    for ch in ch_cols:
        uuid = ch_uuid.get(ch)
        if not uuid or uuid not in sample_neutral_genes:
            continue

        for gene in sample_neutral_genes[uuid]:
            if gene_col:
                gene_rows = abund[abund[gene_col] == gene]
            else:
                gene_rows = abund[abund.index.str.contains(f"GN={gene} ", na=False)]
            if gene_rows.empty:
                continue

            mut_val = gene_rows.iloc[0][ch]
            wt_chs_gene = [c for c in ch_cols if c != ch and
                           gene not in sample_neutral_genes.get(ch_uuid.get(c, ""), set())]
            if pd.isna(mut_val) or not wt_chs_gene:
                continue

            wt_vals = gene_rows.iloc[0][wt_chs_gene].dropna().values
            if len(wt_vals) < 2:
                continue

            records_neutral.append({
                "plex_id":   plex_id,
                "gene":      gene,
                "mut_uuid":  uuid,
                "channel":   ch,
                "mut_abund": mut_val,
                "wt_mean":   np.mean(wt_vals),
                "wt_n":      len(wt_vals),
                "delta":     mut_val - np.mean(wt_vals),
            })

df_neutral = pd.DataFrame(records_neutral)
print(f"Neutral records: {len(df_neutral):,}")
t_n, p_n = stats.ttest_1samp(df_neutral["delta"].dropna(), popmean=0)
print(f"Neutral global t-test: t={t_n:.3f}, p={p_n:.2e}, mean delta={df_neutral['delta'].mean():.4f}")